In [ ]:
import pandas as pd
df = pd.read_csv("/content/IMDB Dataset.csv",encoding ="latin1")
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [ ]:
import re
negation_words = [
    "not good", "not bad", "not great", "don't like "
    "didn't like", "never liked", "wasn't good",
    "isn't good", "no good"
]

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zA-Z\s']"," ", text)

  for phrase in negation_words:
    text = text.replace(phrase, phrase.replace(" ", "_")) # Changed to underscore for clarity

  return text

In [ ]:
df['review'] = df['review'].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad = pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dropout,Dense

model = Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128, dropout = 0.3,recurrent_dropout = 0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')

])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(X_train_pad,y_train,epochs=5,batch_size=64,validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 396s 782ms/step - accuracy: 0.5509 - loss: 0.6697 - val_accuracy: 0.5685 - val_loss: 0.6516
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 389s 778ms/step - accuracy: 0.6246 - loss: 0.6037 - val_accuracy: 0.7516 - val_loss: 0.5633
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 401s 803ms/step - accuracy: 0.7689 - loss: 0.5083 - val_accuracy: 0.7545 - val_loss: 0.5891
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 428s 857ms/step - accuracy: 0.8256 - loss: 0.4234 - val_accuracy: 0.8326 - val_loss: 0.4292
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 430s 859ms/step - accuracy: 0.8256 - loss: 0.4158 - val_accuracy: 0.8549 - val_loss: 0.3881


In [16]:
loss,acc = model.evaluate(X_test_pad,y_test)

print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 103ms/step - accuracy: 0.8556 - loss: 0.3860
Test Accuracy: 0.8555999994277954


In [18]:
def predict_sentiment(review):
  review = clean_text(review)
  seq = tokenizer.texts_to_sequences([review])
  padded = pad_sequences(seq,maxlen=max_len,padding='post')
  prediction = model.predict(padded)[0][0]

  print("\nReview:",review)
  print("Score:",prediction)

  if prediction >= 0.5:
    print("Sentiment: Positive 😍")
  else:
    print("Sentiment: Negative 🤦‍♂️")


In [20]:
predict_sentiment("not super movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step

Review: not super movie
Score: 0.46786875
Sentiment: Negative 🤦‍♂️
